# STG 5.2 — Features de autoría (spec 011)

Construye las features de autoría de cada proyecto: `bloque_autor`, `es_poder_ejecutivo`,
`es_oficialista` y `coincide_bloque_autor` (más las variantes del escenario B para la
duda PRO/LLA), resolviendo el bloque de cada autor con una **cascada de 4 pasos**:

1. **Ejecutivo**: el autor es firmante de un período presidencial y la coalición
   gobernante a la fecha de la votación es la misma de ese período.
2. **Tabla manual**: `tabla_autor_bloque_manual.csv` (pisa al paso 3 si hay conflicto).
3. **Histórico**: bloque del autor como diputado, en la fecha más cercana **anterior o
   igual** a la votación (`merge_asof` hacia atrás — sin mirar el futuro).
4. **Pendiente**: queda listado en `autores_pendientes.csv` para completado manual.

Se ubica como sub-etapa **5.2** porque es enriquecimiento de features sobre `df_modelado`
(como STG_4), y debe correr antes de STG_6 (modelado) — no después de STG_8 (artefactos).

**Entradas**: `df_modelado.csv`, `hcdn_votaciones_historico.csv`, `titulos_autor.xlsx`,
`tabla_periodos_presidenciales.csv`, `tabla_autor_bloque_manual.csv`.

**Salidas**: `df_modelado_autor.csv`, `tabla_autor_bloque.csv`, `autores_pendientes.csv`.

**Reproducibilidad**: este notebook no tiene ninguna fuente de azar (no hace falta
`random_state`). Re-ejecutarlo con los mismos insumos da exactamente el mismo resultado.

In [ ]:
# T4 — Imports y normalización de nombres (mismo criterio que el resto del proyecto)
import pandas as pd
import unicodedata

RUTA_DATA = '../data/'

def normalizar_nombre(s):
    """Mayúsculas, sin tildes (Ñ->N), sin caracteres de encoding roto, espacios colapsados."""
    if pd.isna(s):
        return ''
    s = str(s).upper().strip()
    s = s.replace('\ufffd', 'N')  # caracter roto: PE?A -> PENA
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return ' '.join(s.split())

assert normalizar_nombre('  Peña,  Marcos ') == 'PENA, MARCOS'
assert normalizar_nombre('PE\ufffdA, MARCOS') == 'PENA, MARCOS'
assert normalizar_nombre('Lousteau, Martín') == 'LOUSTEAU, MARTIN'
assert normalizar_nombre(None) == ''
print('normalizar_nombre: 4/4 pruebas OK')

In [ ]:
# T4 — Carga de los 6 insumos (solo lectura: los datos crudos no se tocan)
df_modelado = pd.read_csv(RUTA_DATA + 'df_modelado.csv', parse_dates=['fecha_votacion'])

historico = pd.read_csv(RUTA_DATA + 'hcdn_votaciones_historico.csv',
                        usecols=['DIPUTADO', 'BLOQUE', 'fecha_votacion'])
historico['fecha_votacion'] = pd.to_datetime(historico['fecha_votacion'], errors='coerce')

titulos_autor = pd.read_excel(RUTA_DATA + 'titulos_autor.xlsx')

periodos = pd.read_csv(RUTA_DATA + 'tabla_periodos_presidenciales.csv',
                       parse_dates=['fecha_inicio', 'fecha_fin'])

manual = pd.read_csv(RUTA_DATA + 'tabla_autor_bloque_manual.csv')

# Spec 012: títulos de 2019+ completados a mano (autor + bloque) por el equipo, que antes
# quedaban SIN_DATO. Se identifican por id_votacion, no por el texto del título (el Excel
# tiene caracteres con encoding roto — el id lo esquiva de raíz).
titulos_autor_manual = pd.read_excel(RUTA_DATA + 'titulos_autor_manual.xlsx')
titulos_autor_manual['fecha_votacion'] = pd.to_datetime(titulos_autor_manual['fecha_votacion'], errors='coerce')

print(f'df_modelado:          {df_modelado.shape[0]:,} filas x {df_modelado.shape[1]} cols')
print(f'historico:            {historico.shape[0]:,} filas | diputados únicos: {historico["DIPUTADO"].nunique():,}')
print(f'titulos_autor:        {titulos_autor.shape[0]:,} títulos')
print(f'periodos:             {periodos.shape[0]} períodos presidenciales')
print(f'tabla manual (autor): {manual.shape[0]} autores cargados a mano')
print(f'titulos_autor_manual: {titulos_autor_manual.shape[0]} filas (2019+, completadas a mano — spec 012)')

In [ ]:
# T4 — Validación de formato de las tablas manuales
cols_p = ['presidente', 'fecha_inicio', 'fecha_fin', 'coalicion',
          'firmantes_ejecutivo', 'bloques_integrantes', 'comentario']
assert list(periodos.columns) == cols_p, f'Columnas inesperadas en periodos: {list(periodos.columns)}'
cols_m = ['autor', 'bloque_asignado', 'es_poder_ejecutivo', 'comentario']
assert list(manual.columns) == cols_m, f'Columnas inesperadas en tabla manual: {list(manual.columns)}'

# Fechas parseables, períodos ordenados, sin huecos ni solapamientos
assert periodos['fecha_inicio'].notna().all() and periodos['fecha_fin'].notna().all()
periodos = periodos.sort_values('fecha_inicio').reset_index(drop=True)
for i in range(len(periodos) - 1):
    assert periodos.loc[i, 'fecha_fin'] == periodos.loc[i + 1, 'fecha_inicio'], \
        f'Hueco o solapamiento entre {periodos.loc[i, "presidente"]} y {periodos.loc[i + 1, "presidente"]}'

# Campos clave completos
assert periodos['coalicion'].notna().all()
assert periodos['firmantes_ejecutivo'].notna().all()
assert periodos['bloques_integrantes'].notna().all(), 'Falta completar bloques_integrantes'

# Los períodos cubren todo el rango de votaciones del dataset
assert df_modelado['fecha_votacion'].min() >= periodos['fecha_inicio'].min()
assert df_modelado['fecha_votacion'].max() < periodos['fecha_fin'].max()

print(f'Tablas manuales validadas. Los {len(periodos)} períodos cubren todo el rango de')
print(f'votaciones ({df_modelado["fecha_votacion"].min().date()} a {df_modelado["fecha_votacion"].max().date()}).')

In [ ]:
# T4 — Validación de formato de titulos_autor_manual.xlsx (spec 012)
cols_manual2019 = ['titulo_votacion', 'fecha_votacion', 'id_votacion', 'autor_final',
                   'fuente_autor', 'bloque_autor']
assert list(titulos_autor_manual.columns) == cols_manual2019, \
    f'Columnas inesperadas en titulos_autor_manual.xlsx: {list(titulos_autor_manual.columns)}'

assert titulos_autor_manual['id_votacion'].notna().all(), \
    'Hay id_votacion vacío en titulos_autor_manual.xlsx'
assert titulos_autor_manual['autor_final'].notna().all(), \
    'Hay autor_final vacío en titulos_autor_manual.xlsx'
assert titulos_autor_manual['bloque_autor'].notna().all(), \
    'Hay bloque_autor vacío en titulos_autor_manual.xlsx'
assert pd.api.types.is_integer_dtype(titulos_autor_manual['id_votacion']), \
    f'id_votacion debe ser entero, es {titulos_autor_manual["id_votacion"].dtype}'

print(f'titulos_autor_manual.xlsx validado: {len(titulos_autor_manual)} filas,')
print('sin nulos en id_votacion / autor_final / bloque_autor.')

In [ ]:
# T4 — Autor por título. La fuente de verdad es titulos_autor.xlsx: ahí el equipo va
# completando autores a mano. Un autor cargado a mano (fuente vacía) se marca 'manual_titulo'.
ta = titulos_autor.rename(columns={'titulo_votacion': 'titulo_base'}).copy()
ta['autor_norm'] = ta['autor_final'].map(normalizar_nombre)
ta.loc[ta['autor_final'].notna() & ta['fuente_autor'].isna(), 'fuente_autor'] = 'manual_titulo'

autores_titulo = (ta[ta['autor_final'].notna()]
                  [['titulo_base', 'autor_final', 'autor_norm', 'fuente_autor']]
                  .drop_duplicates('titulo_base'))

# Chequeo de consistencia con df_modelado (los títulos del Excel deben existir allí)
sin_match = ~autores_titulo['titulo_base'].isin(df_modelado['titulo_base'])
assert sin_match.sum() == 0, f'{sin_match.sum()} títulos del Excel no existen en df_modelado'

print(f'Títulos con autor conocido: {len(autores_titulo)} de {ta["titulo_base"].nunique()}')
print()
print('Por fuente del autor:')
print(autores_titulo['fuente_autor'].value_counts().to_string())

In [ ]:
# T4 — Diagnóstico: cuántos autores aparecen en el histórico completo de Diputados.
# Referencia de la spec: 274 de 312. Si crece con el tiempo es porque el equipo completó
# autores nuevos en titulos_autor.xlsx (comportamiento esperado).
dips_hist = set(normalizar_nombre(d) for d in historico['DIPUTADO'].dropna().unique())
en_hist = autores_titulo['autor_norm'].isin(dips_hist)

print(f'Títulos cuyo autor aparece en el histórico de Diputados: {en_hist.sum()} de {len(autores_titulo)}')
print()
print('Autores que NO aparecen (se resolverán por Ejecutivo, tabla manual o pendientes):')
print(autores_titulo[~en_hist]['autor_norm'].value_counts().to_string())

In [ ]:
# T4 — Spec 012: preparar titulos_autor_manual (dedupe + cruce por id_votacion).
# El cruce es por id_votacion (numérico), no por el texto del título: el Excel tiene
# caracteres con encoding roto (mojibake), así que el texto no es una llave confiable.
manual_2019 = titulos_autor_manual.drop_duplicates(subset='id_votacion').copy()
print(f'Deduplicado por id_votacion: {len(titulos_autor_manual)} -> {len(manual_2019)} filas')

ids_dm = df_modelado[['id_votacion', 'titulo_base']].drop_duplicates()
manual_2019 = manual_2019.merge(ids_dm, on='id_votacion', how='left')

sin_match_2019 = manual_2019['titulo_base'].isna()
assert sin_match_2019.sum() == 0, \
    f'{sin_match_2019.sum()} ids de titulos_autor_manual.xlsx no existen en df_modelado'
assert manual_2019['id_votacion'].is_unique, \
    'Debe quedar un id_votacion por fila tras deduplicar'
assert manual_2019.groupby('id_votacion')['titulo_base'].nunique().max() == 1, \
    'Un id_votacion debe mapear a exactamente un titulo_base'

manual_2019['autor_norm'] = manual_2019['autor_final'].map(normalizar_nombre)

# No debe pisar títulos que ya tenían autor conocido (los 312 de la spec 011): hoy los
# 218 títulos de esta tabla son todos SIN_DATO, pero el assert protege corridas futuras
# si se agregan filas nuevas al Excel.
ya_con_autor = manual_2019['titulo_base'].isin(autores_titulo['titulo_base'])
assert ya_con_autor.sum() == 0, \
    f'{ya_con_autor.sum()} títulos de titulos_autor_manual.xlsx ya tenían autor conocido (revisar solapamiento)'

print(f'Cruce contra df_modelado: {manual_2019["titulo_base"].nunique()} títulos únicos, todos SIN_DATO hasta ahora')

In [ ]:
# T4 — Spec 012: normalización y verificación de bloque_autor tipeado a mano.
# El bloque normalizado (mayúsculas/tildes con normalizar_nombre) se compara contra dos
# fuentes: (1) los bloques que existen entre los VOTANTES ACTUALES de df_modelado —
# match directo — y (2) el histórico completo de Diputados, para distinguir un bloque
# real pero sin votantes en el dataset actual de un verdadero error de tipeo. Solo se
# excluyen del chequeo los autores "PEN": su bloque se resuelve por la regla del
# Ejecutivo (T6), no por este mapeo directo.
bloques_dm_norm = {normalizar_nombre(b): b for b in df_modelado['bloque'].dropna().unique()}
bloques_hist_norm = set(normalizar_nombre(b) for b in historico['BLOQUE'].dropna().unique())

manual_2019['bloque_autor_norm'] = manual_2019['bloque_autor'].map(normalizar_nombre)
es_pen_2019 = manual_2019['autor_final'] == 'PEN'

def clasificar_bloque(bn):
    if bn in bloques_dm_norm:
        return 'votantes_actuales'
    if bn in bloques_hist_norm:
        return 'historico_sin_votantes_actuales'
    return 'NO_VERIFICADO'

manual_2019.loc[~es_pen_2019, 'estado_bloque'] = manual_2019.loc[~es_pen_2019, 'bloque_autor_norm'].map(clasificar_bloque)

no_verificados = manual_2019[~es_pen_2019 & (manual_2019['estado_bloque'] == 'NO_VERIFICADO')]
assert len(no_verificados) == 0, (
    f'{len(no_verificados)} bloques cargados a mano no matchean ni a los votantes actuales '
    f'ni al histórico de Diputados (posible error de tipeo): '
    f'{no_verificados["bloque_autor"].unique().tolist()}'
)

resumen_bloques = (manual_2019[~es_pen_2019]
                   .drop_duplicates('bloque_autor')
                   [['bloque_autor', 'estado_bloque']]
                   .sort_values('bloque_autor'))
print('Verificación de los bloques cargados a mano (autores legisladores, no PEN):')
print(resumen_bloques.to_string(index=False))
print()
print(resumen_bloques['estado_bloque'].value_counts().to_string())

In [ ]:
# T5 (spec 012) — Sumar los 218 títulos de titulos_autor_manual a autores_titulo, con
# fuente 'manual_2019'. A partir de acá, estos títulos entran a la misma cascada de
# resolución de bloque (T6/T7) que los 312 originales — sin código separado.
nuevos_autores = (manual_2019[['titulo_base', 'autor_final', 'autor_norm']]
                  .assign(fuente_autor='manual_2019'))

assert not nuevos_autores['titulo_base'].duplicated().any(), \
    'No debería haber titulo_base duplicado entre los títulos manuales 2019+'

antes = len(autores_titulo)
autores_titulo = pd.concat([autores_titulo, nuevos_autores], ignore_index=True)
assert len(autores_titulo) == antes + len(nuevos_autores)
assert autores_titulo['titulo_base'].is_unique, \
    'titulo_base debe seguir siendo único tras sumar los autores manuales 2019+'

print(f'Títulos con autor conocido: {antes} -> {len(autores_titulo)} '
      f'(+{len(nuevos_autores)} de titulos_autor_manual.xlsx)')
print()
print('Por fuente del autor:')
print(autores_titulo['fuente_autor'].value_counts().to_string())

## T5 — Tabla de lookup histórico (bloque de cada diputado por fecha)

Una fila por (diputado, fecha) con su bloque en ese momento. Sirve para responder "¿qué
bloque tenía este diputado en tal fecha?" — el ingrediente central del paso 3 de la
cascada (T7). El lookup es **siempre hacia atrás**: para una fecha de votación dada, se
busca el registro más reciente **anterior o igual**, nunca uno posterior (eso sería mirar
el futuro).

In [ ]:
# T5 — Construir la tabla de lookup: (diputado_norm, fecha) único con su bloque,
# ordenada por fecha (requisito de merge_asof).
lookup_bloque = (
    historico
    .assign(diputado_norm=historico['DIPUTADO'].map(normalizar_nombre))
    .dropna(subset=['fecha_votacion'])
    [['diputado_norm', 'fecha_votacion', 'BLOQUE']]
    .rename(columns={'BLOQUE': 'bloque_historico'})
    .drop_duplicates()
    .sort_values('fecha_votacion')
    .reset_index(drop=True)
)

print(f'Tabla de lookup: {len(lookup_bloque):,} filas (diputado, fecha, bloque)')
print(f'Diputados únicos: {lookup_bloque["diputado_norm"].nunique():,}')

def bloque_a_fecha(autor_norm, fecha):
    """Bloque del autor en el registro más reciente anterior o igual a `fecha`. None si no hay."""
    consulta = pd.DataFrame({'diputado_norm': [autor_norm], 'fecha_votacion': [pd.Timestamp(fecha)]})
    r = pd.merge_asof(consulta.sort_values('fecha_votacion'), lookup_bloque,
                      on='fecha_votacion', by='diputado_norm', direction='backward')
    return r['bloque_historico'].iloc[0]

In [ ]:
# T5 — Test: MASSA, SERGIO TOMAS cambió de bloque varias veces en el histórico.
# Frente Renovador (2014-02 a 2015-11) -> Federal Unidos por una Nueva Argentina
# (2016-01 a 2017-09) -> Frente De Todos (2020-01 a 2022-11). El lookup debe devolver
# el bloque vigente en cada fecha consultada, nunca uno posterior.
massa = 'MASSA, SERGIO TOMAS'
casos = [
    ('2015-06-01', 'Frente Renovador'),
    ('2016-06-01', 'Federal Unidos por una Nueva Argentina'),
    ('2021-01-01', 'Frente De Todos'),
]
for fecha, esperado in casos:
    obtenido = bloque_a_fecha(massa, fecha)
    assert obtenido == esperado, f'{fecha}: esperaba "{esperado}", obtuve "{obtenido}"'
    print(f'{fecha} -> {obtenido}  OK')

# Antes de que Massa apareciera como diputado en el histórico: no hay bloque (None),
# nunca debe "adivinar" mirando el futuro.
antes = bloque_a_fecha(massa, '2013-01-01')
assert pd.isna(antes), f'Se esperaba None/NaN para fecha anterior al histórico, se obtuvo {antes}'
print(f'2013-01-01 -> {antes}  OK (sin dato, no mira el futuro)')

## T6 — Paso 1 de la cascada: detección del Poder Ejecutivo

Un proyecto es "del Ejecutivo" si su autor figura como firmante de **algún** período
presidencial **y** la coalición que gobierna **a la fecha de la votación** es la misma
coalición de ese período (decisión registrada en `memoria/DECISIONES.md`: importa qué
gobierno impulsa el proyecto al momento del voto, no el cargo puntual del firmante — por
eso un título de Alberto Fernández como JGM, votado años después bajo CFK, sigue siendo
Ejecutivo porque ambos períodos comparten la coalición Frente para la Victoria).

Agrupar los firmantes por nombre de coalición (no por período) implementa esta regla de
forma directa: si dos períodos comparten coalición (Néstor Kirchner y CFK, ambos FPV), sus
firmantes se unen automáticamente en el mismo conjunto.

In [ ]:
# T6 — Adjuntar fecha_votacion a autores_titulo (una fecha por título, ya verificado
# en el diagnóstico de T4 que titulo_base es la unidad de agrupación de un solo evento).
fechas_por_titulo = df_modelado[['titulo_base', 'fecha_votacion']].drop_duplicates('titulo_base')
assert fechas_por_titulo['titulo_base'].is_unique, 'Hay títulos con más de una fecha de votación'
autores_titulo = autores_titulo.merge(fechas_por_titulo, on='titulo_base', how='left')
assert autores_titulo['fecha_votacion'].notna().all()

# Firmantes por coalición: explota firmantes_ejecutivo y agrupa por nombre de coalición
# (no por período), para que dos períodos con la misma coalición compartan firmantes.
firmantes_expandido = periodos.assign(
    firmante=periodos['firmantes_ejecutivo'].str.split(';')
).explode('firmante')
firmantes_expandido['firmante_norm'] = firmantes_expandido['firmante'].map(normalizar_nombre)

coalicion_firmantes = (
    firmantes_expandido.groupby('coalicion')['firmante_norm'].apply(set).to_dict()
)

print('Firmantes por coalición:')
for coalicion, firmantes in coalicion_firmantes.items():
    print(f'  {coalicion}: {len(firmantes)} firmantes')

# Coalición gobernante a la fecha de cada título con autor: mismo patrón merge_asof que
# el lookup histórico de T5 (el período vigente es el más reciente que empezó antes o en
# la fecha de la votación). Se lleva también bloques_integrantes del período específico
# (no agrupado por coalición: NK y CFK comparten coalición FPV pero tienen aliados
# distintos, así que cada período conserva su propia lista).
periodos_ord = periodos.sort_values('fecha_inicio')
autores_titulo = pd.merge_asof(
    autores_titulo.sort_values('fecha_votacion'),
    periodos_ord[['fecha_inicio', 'coalicion', 'presidente', 'bloques_integrantes']],
    left_on='fecha_votacion', right_on='fecha_inicio', direction='backward'
).rename(columns={'coalicion': 'coalicion_gobernante', 'presidente': 'presidente_gobernante'})

autores_titulo['es_firmante_coalicion_actual'] = autores_titulo.apply(
    lambda r: r['autor_norm'] in coalicion_firmantes.get(r['coalicion_gobernante'], set()), axis=1
)

n_ejecutivo = autores_titulo['es_firmante_coalicion_actual'].sum()
print()
print(f'Títulos resueltos como Ejecutivo (paso 1): {n_ejecutivo} de {len(autores_titulo)}')

In [ ]:
# T6 — Tests de control sobre casos reales del dataset.

# CFK: los 48 títulos votados 2009-2015, bajo su propia presidencia, dan Ejecutivo.
cfk_titulos = autores_titulo[autores_titulo['autor_norm'] == 'FERNANDEZ DE KIRCHNER, CRISTINA']
assert len(cfk_titulos) == 48
assert cfk_titulos['es_firmante_coalicion_actual'].all(), 'Todos los títulos de CFK deben dar Ejecutivo'
print(f'CFK: {len(cfk_titulos)}/{len(cfk_titulos)} títulos dan Ejecutivo  OK')

# Macri votado en 2026, bajo el gobierno de Milei (LLA != Cambiemos): NO debe dar Ejecutivo.
macri_2026 = autores_titulo[(autores_titulo['autor_norm'] == 'MACRI, MAURICIO') &
                            (autores_titulo['fecha_votacion'] >= '2020-01-01')]
assert len(macri_2026) >= 1
assert not macri_2026['es_firmante_coalicion_actual'].any(), 'Macri votado bajo Milei NO debe ser Ejecutivo'
print(f'Macri (título de {macri_2026["fecha_votacion"].iloc[0].date()}, bajo Milei) -> NO Ejecutivo  OK')

# Alberto Fernández (firmante de N.Kirchner como JGM) con un título votado en 2013, bajo
# CFK: misma coalición (Frente para la Victoria) -> SÍ debe dar Ejecutivo.
af_2013 = autores_titulo[(autores_titulo['autor_norm'] == 'FERNANDEZ, ALBERTO') &
                         (autores_titulo['fecha_votacion'].dt.year == 2013)]
assert len(af_2013) >= 1
assert af_2013['es_firmante_coalicion_actual'].all(), 'A. Fernández 2013 bajo CFK (misma coalición FPV) debe ser Ejecutivo'
print(f'A. Fernández (título de {af_2013["fecha_votacion"].iloc[0].date()}, bajo CFK, misma coalición FPV) -> Ejecutivo  OK')

# Massa (firmante solo del período CFK/FPV como JGM) con un título votado en 2016, bajo
# Macri (Cambiemos != FPV): NO debe dar Ejecutivo.
massa_2016 = autores_titulo[(autores_titulo['autor_norm'] == 'MASSA, SERGIO TOMAS') &
                            (autores_titulo['fecha_votacion'].dt.year == 2016)]
assert len(massa_2016) >= 1
assert not massa_2016['es_firmante_coalicion_actual'].any(), 'Massa 2016 bajo Macri (coalición distinta) NO debe ser Ejecutivo'
print(f'Massa (título de {massa_2016["fecha_votacion"].iloc[0].date()}, bajo Macri, coalición distinta) -> NO Ejecutivo  OK')

In [ ]:
# T6 (spec 012) — Autores "PEN": el equipo ya confirmó que el proyecto lo mandó
# directamente el Ejecutivo, así que no hace falta identificar al firmante puntual (el
# match por nombre de arriba nunca reconocería el literal "PEN"). Se marca directamente
# como Ejecutivo (entra al paso 1 de la cascada), con bloque_autor = coalición
# gobernante a la fecha de la votación (ya calculada en la celda anterior vía
# merge_asof). El bloque que el equipo tipeó a mano en el Excel se usa solo como
# verificación cruzada: ante discrepancia, manda la tabla de períodos presidenciales
# (validada fila por fila por el equipo en la spec 011).
es_pen = (autores_titulo['fuente_autor'] == 'manual_2019') & (autores_titulo['autor_final'] == 'PEN')
autores_titulo.loc[es_pen, 'es_firmante_coalicion_actual'] = True

pen_manual = manual_2019.loc[manual_2019['autor_final'] == 'PEN', ['titulo_base', 'bloque_autor']]
pen_check = (autores_titulo.loc[es_pen, ['titulo_base', 'coalicion_gobernante']]
            .merge(pen_manual, on='titulo_base'))
pen_check['coincide'] = (pen_check['coalicion_gobernante'].map(normalizar_nombre) ==
                        pen_check['bloque_autor'].map(normalizar_nombre))

discrepancias_pen = pen_check[~pen_check['coincide']]
print(f'Autores "PEN": {es_pen.sum()} títulos marcados como Ejecutivo (paso 1 de la cascada)')
print(f'Verificación cruzada con el bloque cargado a mano: '
      f'{len(pen_check) - len(discrepancias_pen)} de {len(pen_check)} coinciden')
if len(discrepancias_pen) > 0:
    print()
    print('Discrepancias (manda la coalición de tabla_periodos_presidenciales.csv):')
    print(discrepancias_pen[['titulo_base', 'coalicion_gobernante', 'bloque_autor']].to_string(index=False))
else:
    print('Sin discrepancias.')

## T7 — Pasos 2 a 4 de la cascada: manual, histórico y pendiente

Orden de prioridad final (la tabla manual tiene **máxima prioridad**: es el mecanismo de
corrección del equipo, así que pisa cualquier asignación automática, incluida la del
Ejecutivo):

1. **Manual** — si el autor está en `tabla_autor_bloque_manual.csv`, se usa esa asignación.
2. **Ejecutivo** — el resultado de T6, para los autores que la tabla manual no cubrió.
3. **Histórico** — lookup temporal hacia atrás (T5), para los que quedan sin resolver.
4. **Pendiente** — `PENDIENTE_MANUAL`, para completar a mano en la próxima corrida.

Los títulos sin autor conocido (710 de 1022) reciben `SIN_DATO`, distinguible de
`PENDIENTE_MANUAL` porque en ese caso ni siquiera se conoce el nombre del autor.

In [ ]:
# T7 — Paso 1 (manual, máxima prioridad): adjuntar la asignación manual a cada título.
manual_norm = manual.copy()
manual_norm['autor_norm'] = manual_norm['autor'].map(normalizar_nombre)

autores_titulo = autores_titulo.merge(
    manual_norm[['autor_norm', 'bloque_asignado', 'es_poder_ejecutivo']]
    .rename(columns={'es_poder_ejecutivo': 'es_poder_ejecutivo_manual'}),
    on='autor_norm', how='left'
)

autores_titulo['bloque_autor'] = pd.NA
autores_titulo['fuente_bloque_autor'] = pd.NA

mask_manual = autores_titulo['bloque_asignado'].notna()
autores_titulo.loc[mask_manual, 'bloque_autor'] = autores_titulo.loc[mask_manual, 'bloque_asignado']
autores_titulo.loc[mask_manual, 'fuente_bloque_autor'] = 'manual'

print(f'Resueltos por tabla manual: {mask_manual.sum()} de {len(autores_titulo)}')

In [ ]:
# T7 (spec 012) — Paso "manual" para los autores legisladores de titulos_autor_manual
# (los que no son "PEN"): el bloque cargado a mano es específico de ESE título, así que
# tiene la misma prioridad que el paso 1 de arriba (tabla_autor_bloque_manual.csv, por
# autor) — de hecho lo pisa si hay conflicto, porque acá se conoce el bloque exacto del
# proyecto puntual, no solo el último bloque conocido del autor en general.
es_legislador_2019 = (autores_titulo['fuente_autor'] == 'manual_2019') & (autores_titulo['autor_final'] != 'PEN')
bloque_por_titulo_2019 = manual_2019.set_index('titulo_base')['bloque_autor']

# Reporte de consistencia: si el autor también está resuelto por la tabla manual
# por-autor (paso 1) con un bloque distinto, se avisa pero se conserva la asignación
# específica del título.
conflicto = autores_titulo[es_legislador_2019 & autores_titulo['fuente_bloque_autor'].eq('manual')]
if len(conflicto) > 0:
    comp = conflicto[['titulo_base', 'autor_final', 'bloque_autor']].rename(
        columns={'bloque_autor': 'bloque_por_autor_ya_asignado'})
    comp['bloque_especifico_del_titulo'] = comp['titulo_base'].map(bloque_por_titulo_2019)
    print('Autores con asignación por-autor (tabla_autor_bloque_manual.csv) — se conserva')
    print('la asignación específica del título:')
    print(comp.to_string(index=False))
    print()
else:
    print('Sin conflicto con la tabla manual por-autor existente.')

autores_titulo.loc[es_legislador_2019, 'bloque_autor'] = autores_titulo.loc[es_legislador_2019, 'titulo_base'].map(bloque_por_titulo_2019)
autores_titulo.loc[es_legislador_2019, 'fuente_bloque_autor'] = 'manual'

print(f'Resueltos por bloque manual específico del título (legisladores, spec 012): {es_legislador_2019.sum()}')

In [ ]:
# T7 — Paso 2 (Ejecutivo): resultado de T6, solo para los que la tabla manual no cubrió.
mask_ejecutivo = autores_titulo['fuente_bloque_autor'].isna() & autores_titulo['es_firmante_coalicion_actual']
autores_titulo.loc[mask_ejecutivo, 'bloque_autor'] = autores_titulo.loc[mask_ejecutivo, 'coalicion_gobernante']
autores_titulo.loc[mask_ejecutivo, 'fuente_bloque_autor'] = 'ejecutivo'

print(f'Resueltos por Ejecutivo: {mask_ejecutivo.sum()}')
print(f'Total resuelto hasta ahora (manual + ejecutivo): {autores_titulo["fuente_bloque_autor"].notna().sum()} de {len(autores_titulo)}')

In [ ]:
# T7 — Paso 3 (histórico): lookup temporal hacia atrás para los que siguen sin resolver.
# Vectorizado con merge_asof (mismo criterio que T5, aplicado a todos los pendientes de una vez).
mask_pendiente_hist = autores_titulo['fuente_bloque_autor'].isna()
subset3 = (autores_titulo.loc[mask_pendiente_hist, ['titulo_base', 'autor_norm', 'fecha_votacion']]
          .sort_values('fecha_votacion'))

hist_resultado = pd.merge_asof(
    subset3, lookup_bloque, on='fecha_votacion',
    left_by='autor_norm', right_by='diputado_norm', direction='backward'
)

autores_titulo = autores_titulo.merge(
    hist_resultado[['titulo_base', 'bloque_historico']], on='titulo_base', how='left'
)

mask_historico = mask_pendiente_hist & autores_titulo['bloque_historico'].notna()
autores_titulo.loc[mask_historico, 'bloque_autor'] = autores_titulo.loc[mask_historico, 'bloque_historico']
autores_titulo.loc[mask_historico, 'fuente_bloque_autor'] = 'historico'

print(f'Resueltos por histórico: {mask_historico.sum()}')
print(f'Total resuelto (manual + ejecutivo + histórico): {autores_titulo["fuente_bloque_autor"].notna().sum()} de {len(autores_titulo)}')

In [ ]:
# T7 — Paso 4 (pendiente): lo que sigue sin resolver queda marcado para completado manual.
mask_pendiente_final = autores_titulo['fuente_bloque_autor'].isna()
autores_titulo.loc[mask_pendiente_final, 'bloque_autor'] = 'PENDIENTE_MANUAL'
autores_titulo.loc[mask_pendiente_final, 'fuente_bloque_autor'] = 'pendiente'

assert autores_titulo['bloque_autor'].notna().all()
assert autores_titulo['fuente_bloque_autor'].notna().all()

print(f'Pendientes de completado manual: {mask_pendiente_final.sum()}')
print()
print('Resumen de los 312 títulos con autor, por fuente de bloque_autor:')
print(autores_titulo['fuente_bloque_autor'].value_counts().to_string())

In [ ]:
# T7 — Tabla completa por título (los 1022 de df_modelado): los que no tienen autor
# conocido reciben SIN_DATO. Esta tabla es la base que T8 va a extender con las otras
# tres features, y T10 va a exportar.
titulos_todos = df_modelado[['titulo_base']].drop_duplicates()
tabla_bloque_por_titulo = titulos_todos.merge(
    autores_titulo[['titulo_base', 'autor_final', 'autor_norm', 'fuente_autor',
                    'bloque_autor', 'fuente_bloque_autor', 'coalicion_gobernante',
                    'bloques_integrantes', 'es_poder_ejecutivo_manual']],
    on='titulo_base', how='left'
)

mask_sin_autor = tabla_bloque_por_titulo['autor_final'].isna()
tabla_bloque_por_titulo.loc[mask_sin_autor, 'bloque_autor'] = 'SIN_DATO'
tabla_bloque_por_titulo.loc[mask_sin_autor, 'fuente_bloque_autor'] = 'sin_dato'

assert tabla_bloque_por_titulo['bloque_autor'].notna().all()
assert tabla_bloque_por_titulo['fuente_bloque_autor'].notna().all()
assert len(tabla_bloque_por_titulo) == titulos_todos['titulo_base'].nunique()

# Resumen por título y por filas (un título se repite una vez por cada diputado que votó).
filas_por_titulo = df_modelado['titulo_base'].value_counts().rename('n_filas')
resumen_fuente = (
    tabla_bloque_por_titulo.merge(filas_por_titulo, left_on='titulo_base', right_index=True)
    .groupby('fuente_bloque_autor')
    .agg(n_titulos=('titulo_base', 'nunique'), n_filas=('n_filas', 'sum'))
    .sort_values('n_titulos', ascending=False)
)

print(f'Total de títulos: {len(tabla_bloque_por_titulo)}')
print()
print('Resumen por fuente (títulos y filas de df_modelado que representan):')
print(resumen_fuente.to_string())

## T8 — Derivar las tres features binarias (+ escenario B)

Con `bloque_autor` resuelto (T7), quedan tres preguntas por responder para cada título:

- **`es_poder_ejecutivo`**: ¿lo mandó directamente el Ejecutivo? 1 si `fuente_bloque_autor`
  es `ejecutivo`, o si la tabla manual lo marcó explícitamente. 0 si se conoce el autor y
  no es el caso. **-1** ("sin dato") si el autor no se conoce o sigue pendiente.
- **`es_oficialista`**: ¿la autoría es del oficialismo del momento? 1 si
  `es_poder_ejecutivo`=1, o si `bloque_autor` integra la coalición gobernante a la fecha
  (columna `bloques_integrantes` del período). Mismo -1 para sin dato.
- **`coincide_bloque_autor`**: ¿el diputado que vota es del mismo espacio que el autor?
  Se calcula **por fila** de `df_modelado` (no por título), porque depende de quién vota:
  si el autor es legislador, compara bloques exactos; si es el Ejecutivo, compara si el
  bloque del votante integra la coalición.

**Escenario B** (duda PRO/LLA, spec 011): se recalculan `es_oficialista_b` y
`coincide_bloque_autor_b` agregando el PRO a los bloques integrantes de La Libertad
Avanza. `bloque_autor` y `es_poder_ejecutivo` no cambian entre escenarios — la spec 012
decidirá empíricamente cuál usar comparando F1-macro.

In [ ]:
# T8 — es_poder_ejecutivo: -1 por defecto (sin dato); 0 para autor conocido; 1 para
# Ejecutivo (paso 1 de la cascada) o manual marcado explícitamente como tal.
CONOCIDO = ['manual', 'ejecutivo', 'historico']

tabla_bloque_por_titulo['es_poder_ejecutivo'] = -1
mask_conocido = tabla_bloque_por_titulo['fuente_bloque_autor'].isin(CONOCIDO)
tabla_bloque_por_titulo.loc[mask_conocido, 'es_poder_ejecutivo'] = 0

mask_pe = (
    (tabla_bloque_por_titulo['fuente_bloque_autor'] == 'ejecutivo') |
    ((tabla_bloque_por_titulo['fuente_bloque_autor'] == 'manual') &
     (tabla_bloque_por_titulo['es_poder_ejecutivo_manual'] == 1))
)
tabla_bloque_por_titulo.loc[mask_pe, 'es_poder_ejecutivo'] = 1

print('es_poder_ejecutivo:')
print(tabla_bloque_por_titulo['es_poder_ejecutivo'].value_counts().sort_index().to_string())

In [ ]:
# T8 — es_oficialista (escenario A) y escenario B (PRO agregado a LLA).
def set_bloques(cadena):
    """Convierte 'Bloque A;Bloque B' en un set de nombres normalizados. Vacío si no hay dato."""
    if pd.isna(cadena):
        return set()
    return set(normalizar_nombre(b) for b in cadena.split(';'))

tabla_bloque_por_titulo['bloques_integrantes_norm'] = (
    tabla_bloque_por_titulo['bloques_integrantes'].map(set_bloques)
)

# Escenario B: mismo set, pero con 'Pro' agregado únicamente en los títulos votados bajo
# la coalición La Libertad Avanza.
mask_lla = tabla_bloque_por_titulo['coalicion_gobernante'] == 'LA LIBERTAD AVANZA'
tabla_bloque_por_titulo['bloques_integrantes_b_norm'] = tabla_bloque_por_titulo['bloques_integrantes_norm']
tabla_bloque_por_titulo.loc[mask_lla, 'bloques_integrantes_b_norm'] = (
    tabla_bloque_por_titulo.loc[mask_lla, 'bloques_integrantes_norm']
    .map(lambda s: s | {normalizar_nombre('Pro')})
)

def calcular_oficialista(fila, columna_bloques):
    if fila['es_poder_ejecutivo'] == -1:
        return -1
    if fila['es_poder_ejecutivo'] == 1:
        return 1
    return int(normalizar_nombre(fila['bloque_autor']) in fila[columna_bloques])

tabla_bloque_por_titulo['es_oficialista'] = tabla_bloque_por_titulo.apply(
    calcular_oficialista, axis=1, columna_bloques='bloques_integrantes_norm'
)
tabla_bloque_por_titulo['es_oficialista_b'] = tabla_bloque_por_titulo.apply(
    calcular_oficialista, axis=1, columna_bloques='bloques_integrantes_b_norm'
)

print('es_oficialista (escenario A, PRO fuera de LLA):')
print(tabla_bloque_por_titulo['es_oficialista'].value_counts().sort_index().to_string())
print()
print('es_oficialista_b (escenario B, PRO dentro de LLA):')
print(tabla_bloque_por_titulo['es_oficialista_b'].value_counts().sort_index().to_string())
print()
diferencia = (tabla_bloque_por_titulo['es_oficialista'] != tabla_bloque_por_titulo['es_oficialista_b']).sum()
print(f'Títulos donde A y B difieren: {diferencia}')

### `coincide_bloque_autor`: se calcula por fila de `df_modelado`

A diferencia de las anteriores, esta feature depende de **quién vota**, no solo del
título. Se arma `df_modelado_autor` uniendo `tabla_bloque_por_titulo` a cada fila de
`df_modelado` por `titulo_base`, y se compara el bloque del votante contra `bloque_autor`
(comparación exacta si el autor es legislador; pertenencia a la coalición si es
Ejecutivo — reutilizando `es_poder_ejecutivo`, que ya distingue ambos casos).

In [ ]:
# T8 — Construir df_modelado_autor: una fila por voto, con las features de autoría unidas.
# Nota: df_modelado ya trae su propia columna 'autor_final' (de la spec 006); no se vuelve
# a traer desde tabla_bloque_por_titulo para no chocar de nombre (pandas la hubiera
# renombrado en silencio a autor_final_x/_y). El campo de autoría fresco que aporta esta
# spec es bloque_autor / fuente_bloque_autor, no autor_final.
df_modelado_autor = df_modelado.merge(
    tabla_bloque_por_titulo[['titulo_base', 'bloque_autor', 'fuente_bloque_autor',
                             'es_poder_ejecutivo', 'es_oficialista', 'es_oficialista_b',
                             'bloques_integrantes_norm', 'bloques_integrantes_b_norm']],
    on='titulo_base', how='left'
)
assert len(df_modelado_autor) == len(df_modelado), 'El merge no debe cambiar la cantidad de filas'
assert 'autor_final' in df_modelado_autor.columns, 'df_modelado_autor debe conservar autor_final tal cual venía en df_modelado'

df_modelado_autor['bloque_votante_norm'] = df_modelado_autor['bloque'].map(normalizar_nombre)
bloque_autor_norm_fila = df_modelado_autor['bloque_autor'].map(normalizar_nombre)

def calcular_coincide(fila, columna_bloques_integrantes):
    if fila['es_poder_ejecutivo'] == -1:
        return -1
    if fila['es_poder_ejecutivo'] == 1:
        return int(fila['bloque_votante_norm'] in fila[columna_bloques_integrantes])
    return int(fila['bloque_votante_norm'] == bloque_autor_norm_fila[fila.name])

df_modelado_autor['coincide_bloque_autor'] = df_modelado_autor.apply(
    calcular_coincide, axis=1, columna_bloques_integrantes='bloques_integrantes_norm'
)
df_modelado_autor['coincide_bloque_autor_b'] = df_modelado_autor.apply(
    calcular_coincide, axis=1, columna_bloques_integrantes='bloques_integrantes_b_norm'
)

print('coincide_bloque_autor (escenario A):')
print(df_modelado_autor['coincide_bloque_autor'].value_counts().sort_index().to_string())
print()
print('coincide_bloque_autor_b (escenario B):')
print(df_modelado_autor['coincide_bloque_autor_b'].value_counts().sort_index().to_string())

## T9 — Chequeos automáticos

Cinco verificaciones que tienen que cumplirse siempre, no solo en esta corrida. Se dejan
como `assert` permanentes (no como prints) para que cualquier corrida futura del notebook
falle ruidosamente si algo se rompe:

(a) todo `es_poder_ejecutivo`=1 implica `es_oficialista`=1.
(b) existen casos de `es_oficialista`=1 con `es_poder_ejecutivo`=0 (legisladores oficialistas).
(c) **anti-leakage**: ninguna asignación `historico` usó un registro posterior a la votación.
(d) los 312 títulos con autor conocido no tienen vacíos silenciosos.
(e) ejemplos concretos de `coincide_bloque_autor` = 1 y = 0, verificados a mano.

In [ ]:
# T9 (a) — todo es_poder_ejecutivo=1 implica es_oficialista=1 (en ambos escenarios).
mask_pe1 = tabla_bloque_por_titulo['es_poder_ejecutivo'] == 1
assert (tabla_bloque_por_titulo.loc[mask_pe1, 'es_oficialista'] == 1).all(), \
    'Hay títulos con es_poder_ejecutivo=1 pero es_oficialista != 1'
assert (tabla_bloque_por_titulo.loc[mask_pe1, 'es_oficialista_b'] == 1).all(), \
    'Hay títulos con es_poder_ejecutivo=1 pero es_oficialista_b != 1'
print(f'(a) OK: los {mask_pe1.sum()} títulos con es_poder_ejecutivo=1 tienen es_oficialista=1')

# T9 (b) — existen legisladores oficialistas (es_oficialista=1 sin ser el Ejecutivo).
mask_leg_oficialista = (tabla_bloque_por_titulo['es_oficialista'] == 1) & (tabla_bloque_por_titulo['es_poder_ejecutivo'] == 0)
assert mask_leg_oficialista.sum() > 0, 'No hay ningún caso de legislador oficialista (es_oficialista=1, es_poder_ejecutivo=0)'
print(f'(b) OK: {mask_leg_oficialista.sum()} títulos son de legisladores oficialistas (no Ejecutivo)')

In [ ]:
# T9 (c) — anti-leakage: ninguna asignación 'historico' usó un registro posterior a la
# votación. El merge_asof de T7 (celda 62232f56) usó on='fecha_votacion' con el MISMO
# nombre de columna en ambos lados, así que el resultado solo conserva la fecha del
# título y pierde la fecha real del registro histórico que matcheó — no alcanza para
# verificar la garantía anti-leakage. Se rehace el merge acá con la fecha del registro
# histórico en una columna con nombre distinto, para poder comparar ambas explícitamente.
lookup_bloque_verif = lookup_bloque.rename(columns={'fecha_votacion': 'fecha_registro_historico'})

titulos_historico = tabla_bloque_por_titulo[tabla_bloque_por_titulo['fuente_bloque_autor'] == 'historico']
subset_verif = titulos_historico[['titulo_base', 'autor_norm']].merge(
    fechas_por_titulo, on='titulo_base'
).sort_values('fecha_votacion')

verif_resultado = pd.merge_asof(
    subset_verif, lookup_bloque_verif,
    left_on='fecha_votacion', right_on='fecha_registro_historico',
    left_by='autor_norm', right_by='diputado_norm', direction='backward'
)

assert verif_resultado['fecha_registro_historico'].notna().all(), \
    'Hay asignaciones historico sin fecha de registro (no debería pasar, ya se filtró por fuente==historico)'
assert (verif_resultado['fecha_registro_historico'] <= verif_resultado['fecha_votacion']).all(), \
    'Se encontró al menos una asignación historico que usa un registro POSTERIOR a la votación (leakage)'

print(f'(c) OK: las {len(verif_resultado)} asignaciones por histórico usan un registro con')
print('fecha anterior o igual a la fecha de la votación (sin mirar el futuro).')

In [ ]:
# T9 (d) — los títulos con autor conocido no tienen vacíos silenciosos: todos con
# bloque_autor y fuente asignados, y la fuente es una de las 4 categorías válidas para
# autor conocido (nunca 'sin_dato', que es exclusiva de títulos sin autor). El total
# esperado (312 de la spec 011 + 218 de la spec 012 = 530) se calcula, no se hardcodea,
# para que la corrida siga siendo válida si el equipo agrega más autores en el futuro.
total_esperado = len(autores_titulo)
con_autor = tabla_bloque_por_titulo[tabla_bloque_por_titulo['autor_final'].notna()]
assert len(con_autor) == total_esperado, \
    f'Se esperaban {total_esperado} títulos con autor, hay {len(con_autor)}'
assert con_autor['bloque_autor'].notna().all(), 'Hay títulos con autor pero sin bloque_autor asignado'
assert con_autor['fuente_bloque_autor'].notna().all(), 'Hay títulos con autor pero sin fuente_bloque_autor'
assert con_autor['fuente_bloque_autor'].isin(['manual', 'ejecutivo', 'historico', 'pendiente']).all(), \
    'Hay títulos con autor conocido marcados sin_dato (no debería pasar)'
print(f"(d) OK: los {len(con_autor)} títulos con autor conocido tienen bloque_autor y fuente asignados,")
print("sin vacíos silenciosos.")

In [ ]:
# T9 (e) — ejemplos concretos de coincide_bloque_autor = 1 y = 0, verificados a mano
# (formaliza como asserts permanentes lo que se comprobó de forma exploratoria en T8).

# Caso Ejecutivo: título de CFK (bloque_autor='FRENTE PARA LA VICTORIA'). Un votante del
# bloque 'Frente para la Victoria - PJ' integra esa coalición -> coincide=1. Un votante de
# la UCR no la integra -> coincide=0.
titulo_cfk = tabla_bloque_por_titulo[
    tabla_bloque_por_titulo['autor_final'].fillna('').str.contains('FERNANDEZ DE KIRCHNER')
].iloc[0]['titulo_base']
filas_cfk = df_modelado_autor[df_modelado_autor['titulo_base'] == titulo_cfk]

votante_fpv = filas_cfk[filas_cfk['bloque'] == 'Frente para la Victoria - PJ']
assert len(votante_fpv) > 0 and (votante_fpv['coincide_bloque_autor'] == 1).all(), \
    'Un votante de Frente para la Victoria - PJ debería coincidir con un título de CFK'

votante_ucr = filas_cfk[filas_cfk['bloque'].str.contains('Radical', case=False, na=False)]
assert len(votante_ucr) > 0 and (votante_ucr['coincide_bloque_autor'] == 0).all(), \
    'Un votante de la UCR NO debería coincidir con un título de CFK'
print('(e.1) OK: título Ejecutivo de CFK — votante FPV coincide=1, votante UCR coincide=0')

# Caso legislador: título con autora del bloque Pro. Un votante del Pro coincide=1; un
# votante de otro bloque no.
titulo_leg = tabla_bloque_por_titulo[tabla_bloque_por_titulo['fuente_bloque_autor'] == 'historico'].iloc[0]
filas_leg = df_modelado_autor[df_modelado_autor['titulo_base'] == titulo_leg['titulo_base']]
bloque_autor_leg_norm = normalizar_nombre(titulo_leg['bloque_autor'])

votante_mismo_bloque = filas_leg[filas_leg['bloque'].map(normalizar_nombre) == bloque_autor_leg_norm]
assert len(votante_mismo_bloque) > 0 and (votante_mismo_bloque['coincide_bloque_autor'] == 1).all(), \
    f'Un votante del mismo bloque que el autor ({titulo_leg["bloque_autor"]}) debería coincidir'

votante_otro_bloque = filas_leg[filas_leg['bloque'].map(normalizar_nombre) != bloque_autor_leg_norm]
assert len(votante_otro_bloque) > 0 and (votante_otro_bloque['coincide_bloque_autor'] == 0).all(), \
    'Un votante de un bloque distinto al del autor NO debería coincidir'
print(f'(e.2) OK: título de {titulo_leg["autor_final"]} ({titulo_leg["bloque_autor"]}) —')
print('votante del mismo bloque coincide=1, votante de otro bloque coincide=0')

## T10 — Exportar salidas

Tres archivos nuevos (nada sobrescribe `df_modelado.csv` ni el histórico crudo), más la
pre-carga de los pendientes en la tabla manual:

1. **`df_modelado_autor.csv`** — `df_modelado` + las 7 columnas nuevas.
2. **`tabla_autor_bloque.csv`** — una fila por autor (no por título), con su bloque más
   reciente conocido. Insumo para que la app traduzca un autor hipotético (spec futura).
3. **`autores_pendientes.csv`** — los 37 títulos pendientes, con contexto para completar.
4. **`tabla_autor_bloque_manual.csv`** actualizada — se agregan los autores pendientes que
   todavía no estén cargados, **sin tocar** filas que el equipo ya haya completado (la
   pre-carga es aditiva e idempotente: correr esto de nuevo no duplica ni pisa nada).

In [ ]:
# T10.1 — df_modelado_autor.csv: df_modelado + las 7 columnas nuevas.
COLUMNAS_NUEVAS = ['bloque_autor', 'fuente_bloque_autor', 'es_poder_ejecutivo',
                   'es_oficialista', 'coincide_bloque_autor',
                   'es_oficialista_b', 'coincide_bloque_autor_b']

df_modelado_autor_export = df_modelado_autor[list(df_modelado.columns) + COLUMNAS_NUEVAS]

assert len(df_modelado_autor_export) == len(df_modelado)
assert list(df_modelado_autor_export.columns[:len(df_modelado.columns)]) == list(df_modelado.columns), \
    'Las columnas originales de df_modelado deben conservarse tal cual, en el mismo orden'
assert df_modelado_autor_export.shape[1] == df_modelado.shape[1] + 7

df_modelado_autor_export.to_csv(RUTA_DATA + 'df_modelado_autor.csv', index=False)
print(f'Exportado: df_modelado_autor.csv — {df_modelado_autor_export.shape[0]:,} filas x {df_modelado_autor_export.shape[1]} columnas')
print(f'Columnas nuevas: {COLUMNAS_NUEVAS}')

In [ ]:
# T10.2 — tabla_autor_bloque.csv: una fila por autor (no por título), con el bloque más
# reciente que le conocemos (un autor puede escribir varios proyectos en distintos años;
# nos quedamos con su asignación más reciente, la más relevante para un autor hipotético
# en la app). El diseño detallado del "autor hipotético" en la app queda para una spec
# futura; esto solo deja la tabla de consulta lista.
tabla_autor_bloque = (
    autores_titulo.sort_values('fecha_votacion')
    .groupby('autor_norm')
    .agg(
        autor=('autor_final', 'last'),
        bloque_autor=('bloque_autor', 'last'),
        fuente_bloque_autor=('fuente_bloque_autor', 'last'),
        ultima_fecha_conocida=('fecha_votacion', 'last'),
        n_titulos=('titulo_base', 'nunique'),
    )
    .reset_index()
    .sort_values('autor')
    .drop(columns='autor_norm')
)

assert tabla_autor_bloque['autor'].is_unique
tabla_autor_bloque.to_csv(RUTA_DATA + 'tabla_autor_bloque.csv', index=False)
print(f'Exportado: tabla_autor_bloque.csv — {len(tabla_autor_bloque)} autores únicos')
print()
print(tabla_autor_bloque['fuente_bloque_autor'].value_counts().to_string())

In [ ]:
# T10.3 — autores_pendientes.csv: los títulos que quedaron PENDIENTE_MANUAL, con contexto
# (título, fecha, motivo) para completar la tabla manual.
pendientes = (
    tabla_bloque_por_titulo[tabla_bloque_por_titulo['fuente_bloque_autor'] == 'pendiente']
    [['titulo_base', 'autor_final', 'coalicion_gobernante']]
    .merge(fechas_por_titulo, on='titulo_base')
    .sort_values(['autor_final', 'fecha_votacion'])
)
pendientes['motivo'] = (
    'No es firmante del Ejecutivo en esa fecha, no está en la tabla manual '
    'y no aparece como diputado en el histórico'
)

pendientes.to_csv(RUTA_DATA + 'autores_pendientes.csv', index=False)
print(f'Exportado: autores_pendientes.csv — {len(pendientes)} títulos, {pendientes["autor_final"].nunique()} autores únicos')

In [ ]:
# T10.4 — Pre-cargar tabla_autor_bloque_manual.csv con los autores pendientes (aditivo:
# no toca autores que el equipo ya haya completado en una corrida anterior).
pendientes_autores = pendientes[['autor_final']].drop_duplicates().rename(columns={'autor_final': 'autor'})
pendientes_autores['autor_norm'] = pendientes_autores['autor'].map(normalizar_nombre)

ya_cargados = set(manual_norm['autor_norm'])
nuevos = pendientes_autores[~pendientes_autores['autor_norm'].isin(ya_cargados)]

if len(nuevos) > 0:
    filas_nuevas = pd.DataFrame({
        'autor': nuevos['autor'],
        'bloque_asignado': pd.NA,
        'es_poder_ejecutivo': pd.NA,
        'comentario': 'Pendiente (spec 011) — completar bloque_asignado; es_poder_ejecutivo=1 solo si es un firmante institucional del Ejecutivo (ej. Jefatura de Gabinete)',
    })
    manual_actualizada = pd.concat([manual, filas_nuevas], ignore_index=True)
    manual_actualizada.to_csv(RUTA_DATA + 'tabla_autor_bloque_manual.csv', index=False)
else:
    manual_actualizada = manual

print(f'tabla_autor_bloque_manual.csv: {len(manual)} autores ya cargados + {len(nuevos)} pendientes nuevos = {len(manual_actualizada)} filas totales')